In [1]:
!pip install google-cloud-storage scikit-learn -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mlflow 2.13.2 requires protobuf<5,>=3.12.0, but you have protobuf 7.36.0 which is incompatible.
kfp 2.16.1 requires protobuf<7.0,>=6.31.1, but you have protobuf 7.36.0 which is incompatible.
databricks-sdk 0.122.0 requires protobuf!=5.26.*,!=5.27.*,!=5.28.*,!=5.29.0,!=5.29.1,!=5.29.2,!=5.29.3,!=5.29.4,!=6.30.0,!=6.30.1,!=6.31.0,<7.0,>=4.25.8, but you have protobuf 7.36.0 which is incompatible.
kfp-pipeline-spec 2.16.1 requires protobuf<7.0,>=6.31.1, but you have protobuf 7.36.0 which is incompatible.
feast 0.64.0 requires numpy<3,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
feast 0.64.0 requires pyarrow>=21.0.0; extra != "flink", but you have pyarrow 15.0.2 which is incompatible.
bigframes 2.43.0 requires pyarrow>=23.0.1, but you have pyarrow 15.0.2 which is incompatible.
google-cloud-aiplatform 1.130

In [2]:
import json
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from google.cloud import storage

In [3]:
# Load college provided IRIS dataset
df = pd.read_csv('/home/jupyter/data/iris_clean.csv')

print(df.head())
print(f"\nTotal samples: {len(df)}")
print(f"Species: {df['species'].unique()}")

   sepal_length  sepal_width  petal_length  petal_width species
0           5.8          4.0           1.2          0.2  setosa
1           5.7          4.4           1.5          0.4  setosa
2           5.4          3.9           1.3          0.4  setosa
3           5.1          3.5           1.4          0.3  setosa
4           5.7          3.8           1.7          0.3  setosa

Total samples: 101
Species: ['setosa' 'versicolor' 'virginica']


In [4]:
from sklearn.model_selection import train_test_split

# Split data - 80% train, 20% test
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['species'])

print(f"Train samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")
print(f"\nTrain species distribution:\n{train_df['species'].value_counts()}")
print(f"\nTest species distribution:\n{test_df['species'].value_counts()}")

Train samples: 80
Test samples: 21

Train species distribution:
species
virginica     28
setosa        28
versicolor    24
Name: count, dtype: int64

Test species distribution:
species
setosa        8
virginica     7
versicolor    6
Name: count, dtype: int64


In [10]:
# Create V1 - Raw format JSONL (Gemini format)
v1_gemini = []

for _, row in train_df.iterrows():
    record = {
        "contents": [
            {"role": "user", "parts": [{"text": f"sepal_length: {row['sepal_length']}, sepal_width: {row['sepal_width']}, petal_length: {row['petal_length']}, petal_width: {row['petal_width']}"}]},
            {"role": "model", "parts": [{"text": row['species']}]}
        ]
    }
    v1_gemini.append(record)

with open('iris_v1_train.jsonl', 'w') as f:
    for record in v1_gemini:
        f.write(json.dumps(record) + '\n')

print(f"V1 records: {len(v1_gemini)}")
print(f"\nExample:")
print(json.dumps(v1_gemini[0], indent=2))

V1 records: 80

Example:
{
  "contents": [
    {
      "role": "user",
      "parts": [
        {
          "text": "sepal_length: 6.9, sepal_width: 3.1, petal_length: 5.1, petal_width: 2.3"
        }
      ]
    },
    {
      "role": "model",
      "parts": [
        {
          "text": "virginica"
        }
      ]
    }
  ]
}


In [15]:
# Create V2 - Natural Language format JSONL (Gemini format)
v2_gemini = []

for _, row in train_df.iterrows():
    record = {
        "contents": [
            {"role": "user", "parts": [{"text": f"A flower specimen has a sepal length of {row['sepal_length']} cm, sepal width of {row['sepal_width']} cm, petal length of {row['petal_length']} cm, and petal width of {row['petal_width']} cm. Identify the iris species."}]},
            {"role": "model", "parts": [{"text": f"This is Iris {row['species']}."}]}
        ]
    }
    v2_gemini.append(record)

with open('iris_v2_train.jsonl', 'w') as f:
    for record in v2_gemini:
        f.write(json.dumps(record) + '\n')

print(f"V2 records: {len(v2_gemini)}")
print(f"\nExample:")
print(json.dumps(v2_gemini[0], indent=2))

V2 records: 80

Example:
{
  "contents": [
    {
      "role": "user",
      "parts": [
        {
          "text": "A flower specimen has a sepal length of 6.9 cm, sepal width of 3.1 cm, petal length of 5.1 cm, and petal width of 2.3 cm. Identify the iris species."
        }
      ]
    },
    {
      "role": "model",
      "parts": [
        {
          "text": "This is Iris virginica."
        }
      ]
    }
  ]
}


In [16]:
# Save test data in correct Gemini format
v1_test_gemini = []
v2_test_gemini = []

for _, row in test_df.iterrows():
    # V1 test
    v1_test_gemini.append({
        "contents": [
            {"role": "user", "parts": [{"text": f"sepal_length: {row['sepal_length']}, sepal_width: {row['sepal_width']}, petal_length: {row['petal_length']}, petal_width: {row['petal_width']}"}]},
            {"role": "model", "parts": [{"text": row['species']}]}
        ]
    })
    # V2 test
    v2_test_gemini.append({
        "contents": [
            {"role": "user", "parts": [{"text": f"A flower specimen has a sepal length of {row['sepal_length']} cm, sepal width of {row['sepal_width']} cm, petal length of {row['petal_length']} cm, and petal width of {row['petal_width']} cm. Identify the iris species."}]},
            {"role": "model", "parts": [{"text": f"This is Iris {row['species']}."}]}
        ]
    })

# Save files
with open('iris_v1_test.jsonl', 'w') as f:
    for record in v1_test_gemini:
        f.write(json.dumps(record) + '\n')

with open('iris_v2_test.jsonl', 'w') as f:
    for record in v2_test_gemini:
        f.write(json.dumps(record) + '\n')

test_df.to_csv('iris_test.csv', index=False)

print(f"V1 test records: {len(v1_test_gemini)}")
print(f"V2 test records: {len(v2_test_gemini)}")
print("All test files saved!")

V1 test records: 21
V2 test records: 21
All test files saved!


In [17]:
from google.cloud import storage

# Upload to GCS
def upload_to_gcs(bucket_name, source_file, destination_blob):
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(destination_blob)
    blob.upload_from_filename(source_file)
    print(f" Uploaded {source_file} → gs://{bucket_name}/{destination_blob}")

BUCKET_NAME = "llmops-iris-week10"

# Upload all fixed files
upload_to_gcs(BUCKET_NAME, "iris_v1_train.jsonl", "data/iris_v1_train.jsonl")
upload_to_gcs(BUCKET_NAME, "iris_v2_train.jsonl", "data/iris_v2_train.jsonl")
upload_to_gcs(BUCKET_NAME, "iris_v1_test.jsonl",  "data/iris_v1_test.jsonl")
upload_to_gcs(BUCKET_NAME, "iris_v2_test.jsonl",  "data/iris_v2_test.jsonl")
upload_to_gcs(BUCKET_NAME, "iris_test.csv",        "data/iris_test.csv")

print("\n All fixed files uploaded to GCS!")

 Uploaded iris_v1_train.jsonl → gs://llmops-iris-week10/data/iris_v1_train.jsonl
 Uploaded iris_v2_train.jsonl → gs://llmops-iris-week10/data/iris_v2_train.jsonl
 Uploaded iris_v1_test.jsonl → gs://llmops-iris-week10/data/iris_v1_test.jsonl
 Uploaded iris_v2_test.jsonl → gs://llmops-iris-week10/data/iris_v2_test.jsonl
 Uploaded iris_test.csv → gs://llmops-iris-week10/data/iris_test.csv

 All fixed files uploaded to GCS!


In [18]:
# Verify files in GCS
client = storage.Client()
bucket = client.bucket(BUCKET_NAME)
blobs = bucket.list_blobs(prefix="data/")

print("Files in GCS bucket:")
for blob in blobs:
    print(f"  📄 gs://{BUCKET_NAME}/{blob.name}")

Files in GCS bucket:
  📄 gs://llmops-iris-week10/data/iris_test.csv
  📄 gs://llmops-iris-week10/data/iris_v1_test.jsonl
  📄 gs://llmops-iris-week10/data/iris_v1_train.jsonl
  📄 gs://llmops-iris-week10/data/iris_v2_test.jsonl
  📄 gs://llmops-iris-week10/data/iris_v2_train.jsonl
